# Data Inspection and Horizon Diagnostics

This notebook inspects the raw LOBSTER order-book and message data, checks data
quality and market-session consistency, compares activity and liquidity across
assets, and motivates the common 500 ms calendar-time representation used in
the downstream analysis.

The notebook also examines candidate forecasting horizons using return
distributions and simple predictability diagnostics. These exploratory results
are used to justify the asset-specific horizon choices implemented in the
feature-engineering and modeling notebooks.

In [ ]:
import sys
import numpy as np
import pandas as pd
import pyarrow

print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyArrow:", pyarrow.__version__)

In [ ]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

DATA_DIR = Path("../data/task_data")

asset = "TSLA"
date = "2026-03-18"

day_dir = DATA_DIR / asset / date

message_path = day_dir / f"message_{date}.parquet"
orderbook_path = day_dir / f"orderbook_{date}.parquet"

print("Message path:", message_path)
print("Orderbook path:", orderbook_path)

print("Message exists:", message_path.exists())
print("Orderbook exists:", orderbook_path.exists())

In [ ]:
message_pf = pq.ParquetFile(message_path)
orderbook_pf = pq.ParquetFile(orderbook_path)

print("MESSAGE")
print("Rows:", message_pf.metadata.num_rows)
print("Columns:", message_pf.metadata.num_columns)
print("Row groups:", message_pf.metadata.num_row_groups)
print(message_pf.schema)

print("\nORDERBOOK")
print("Rows:", orderbook_pf.metadata.num_rows)
print("Columns:", orderbook_pf.metadata.num_columns)
print("Row groups:", orderbook_pf.metadata.num_row_groups)
print(orderbook_pf.schema)

In [ ]:
message_df = pd.read_parquet(message_path)
orderbook_df = pd.read_parquet(orderbook_path)

print("Message shape:", message_df.shape)
print("Orderbook shape:", orderbook_df.shape)

display(message_df.head(10))
display(orderbook_df.head(10))

In [ ]:
print("Message columns:")
print(message_df.columns.tolist())

print("\nOrderbook columns:")
print(orderbook_df.columns.tolist())

# 1. Initial Data Quality Inspection

We first inspect timestamps, event types, missing values, duplicates,
price/size ranges, and basic limit-order-book validity for one asset-day.

In [ ]:
print("Raw first timestamp:", message_df["time"].iloc[0])
print("Raw last timestamp: ", message_df["time"].iloc[-1])

time_dt = pd.to_datetime(message_df["time"], unit="ns")

print("\nConverted first timestamp:", time_dt.iloc[0])
print("Converted last timestamp: ", time_dt.iloc[-1])

print("\nDuration:", time_dt.iloc[-1] - time_dt.iloc[0])

print("\nTimestamps monotonically increasing:",
      message_df["time"].is_monotonic_increasing)

print("Duplicate timestamps:",
      message_df["time"].duplicated().sum())

In [ ]:
# Exact duplicate message-row diagnostic
exact_duplicate_messages = message_df.duplicated().sum()

print("Exact duplicate message rows:", exact_duplicate_messages)
print(
    "Exact duplicate message rows (%):",
    100 * exact_duplicate_messages / len(message_df)
)

### Timestamp and Duplicate-Row Check

The message timestamps are monotonically increasing, but many timestamps are
repeated because multiple distinct market events can occur at the same recorded
nanosecond. Therefore, repeated timestamps are not treated as erroneous
duplicates.

To distinguish repeated timestamps from actual duplicated observations, exact
message-row duplicates were also checked across all message fields. No exact duplicate message rows were found for this representative asset-day, so no observations were removed in this initial inspection. A systematic training-set audit across all asset-days is performed below.

In [ ]:
event_counts = message_df["event_type"].value_counts()

print(event_counts)

print("\nPercentages:")
print((event_counts / len(message_df) * 100).round(2))

In [ ]:
print(message_df["direction"].value_counts(dropna=False))

In [ ]:
print("MESSAGE missing values:")
print(message_df.isna().sum())

print("\nORDERBOOK missing values:")
print(orderbook_df.isna().sum())

In [ ]:
print("Message size:")
print(message_df["size"].describe())

print("\nMessage price:")
print(message_df["price"].describe())

In [ ]:
level1_cols = ["sell1", "vsell1", "buy1", "vbuy1"]

display(orderbook_df[level1_cols].describe())

In [ ]:
crossed = orderbook_df["buy1"] > orderbook_df["sell1"]
locked = orderbook_df["buy1"] == orderbook_df["sell1"]

print("Crossed book states:", crossed.sum())
print("Locked book states:", locked.sum())
print("Total rows:", len(orderbook_df))

print("Crossed %:", crossed.mean() * 100)
print("Locked %:", locked.mean() * 100)

In [ ]:
ask_order_violations = 0
bid_order_violations = 0

for level in range(1, 10):
    ask_order_violations += (
        orderbook_df[f"sell{level}"] >
        orderbook_df[f"sell{level+1}"]
    ).sum()

    bid_order_violations += (
        orderbook_df[f"buy{level}"] <
        orderbook_df[f"buy{level+1}"]
    ).sum()

print("Ask ordering violations:", ask_order_violations)
print("Bid ordering violations:", bid_order_violations)

In [ ]:
best_ask = orderbook_df["sell1"]
best_bid = orderbook_df["buy1"]

mid_price = (best_ask + best_bid) / 2
spread = best_ask - best_bid

imbalance_l1 = (
    orderbook_df["vbuy1"] - orderbook_df["vsell1"]
) / (
    orderbook_df["vbuy1"] + orderbook_df["vsell1"]
)

print("Mid-price:")
print(mid_price.describe())

print("\nSpread:")
print(spread.describe())

print("\nL1 imbalance:")
print(imbalance_l1.describe())

In [ ]:
# Which event types are associated with zero or extreme message prices?
special_prices = message_df[
    (message_df["price"] == 0) |
    (message_df["price"] >= 90_000_000)
]

print(special_prices["event_type"].value_counts())
display(special_prices.head(20))

In [ ]:
# Typical message prices by event type
for event in message_df["event_type"].unique():
    subset = message_df.loc[message_df["event_type"] == event, "price"]
    print(f"\n{event}")
    print(subset.describe())

In [ ]:
sample_prices = orderbook_df[["sell1", "buy1"]].iloc[[0, 1000, 100000, -1]].copy()

sample_prices["sell1_div_1e4"] = sample_prices["sell1"] / 1e4
sample_prices["buy1_div_1e4"] = sample_prices["buy1"] / 1e4

display(sample_prices)

## Cross-Asset and Intraday Diagnostics

After inspecting one representative asset-day in detail, the analysis is
extended across all assets and dates. The goal is to compare message intensity,
execution activity, spreads, book behavior, and intraday coverage before
choosing a common temporal representation.

The analysis also checks regular-session coverage and sampling characteristics
across assets to support the calendar-time representation used below.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

DATA_DIR = Path("../data/task_data")

def summarize_day(asset, date):
    day_dir = DATA_DIR / asset / date

    message_path = day_dir / f"message_{date}.parquet"
    orderbook_path = day_dir / f"orderbook_{date}.parquet"

    # Read only the columns needed for summary
    msg = pd.read_parquet(
        message_path,
        columns=["time", "event_type", "size", "price", "direction"]
    )

    ob = pd.read_parquet(
        orderbook_path,
        columns=["sell1", "vsell1", "buy1", "vbuy1"]
    )

    # Convert top-of-book prices to dollars
    ask = ob["sell1"] / 1e4
    bid = ob["buy1"] / 1e4

    mid = (ask + bid) / 2
    spread = ask - bid

    imbalance = (
        ob["vbuy1"] - ob["vsell1"]
    ) / (
        ob["vbuy1"] + ob["vsell1"]
    )

    # Event counts
    counts = msg["event_type"].value_counts()

    # Timestamp information
    first_time = pd.to_datetime(msg["time"].iloc[0], unit="ns")
    last_time = pd.to_datetime(msg["time"].iloc[-1], unit="ns")

    duration_seconds = (
        msg["time"].iloc[-1] - msg["time"].iloc[0]
    ) / 1e9

    # Basic activity rates
    events_per_second = len(msg) / duration_seconds

    executions = counts.get("OrderExecuted", 0)
    executions_per_second = executions / duration_seconds

    return {
        "asset": asset,
        "date": date,
        "rows": len(msg),

        "start": first_time,
        "end": last_time,

        "events_per_sec": events_per_second,
        "executions": executions,
        "executions_per_sec": executions_per_second,

        "add_orders": counts.get("AddOrder", 0),
        "deletes": counts.get("OrderDelete", 0),

        "mid_mean": mid.mean(),
        "mid_std": mid.std(),

        "spread_mean": spread.mean(),
        "spread_median": spread.median(),
        "spread_p95": spread.quantile(0.95),

        "l1_imbalance_mean": imbalance.mean(),
        "l1_imbalance_std": imbalance.std(),

        "avg_bid_size": ob["vbuy1"].mean(),
        "avg_ask_size": ob["vsell1"].mean(),

        "crossed_books": (bid > ask).sum(),
        "locked_books": (bid == ask).sum(),
    }

In [ ]:
assets = ["TSLA", "AMZN", "BRK.B"]
dates = [
    "2026-03-16",
    "2026-03-17",
    "2026-03-18",
]

summaries = []

for asset in assets:
    for date in dates:
        print(f"Processing {asset} {date}...")
        summaries.append(summarize_day(asset, date))

summary_df = pd.DataFrame(summaries)

display(summary_df)

### Training-Set Data Quality Audit

The detailed checks above were first performed on a representative asset-day.
To verify that the conclusions are not driven by that single sample, a compact
automated audit is repeated across all training asset-days. The audit checks
row alignment, timestamp ordering, exact duplicate rows, missing values, price
and size validity, and abnormal limit-order-book states.

In [ ]:
# ============================================================
# Training-set raw data quality audit
# Assets: TSLA, AMZN, BRK.B
# Dates : 2026-03-16 to 2026-03-18
#
# This is a diagnostic audit only. It does NOT modify any data.
# Checks are performed within the regular trading session
# (09:30 <= time < 16:00, America/New_York).
# ============================================================

import gc

def audit_training_day(asset, date):
    day_dir = DATA_DIR / asset / date

    message_path = day_dir / f"message_{date}.parquet"
    orderbook_path = day_dir / f"orderbook_{date}.parquet"

    # Load one asset-day at a time to keep memory usage manageable
    msg = pd.read_parquet(message_path)
    ob = pd.read_parquet(orderbook_path)

    # --------------------------------------------------------
    # 1. Basic row alignment
    # --------------------------------------------------------
    message_rows_raw = len(msg)
    orderbook_rows_raw = len(ob)
    row_count_match = message_rows_raw == orderbook_rows_raw

    # --------------------------------------------------------
    # 2. Timestamp checks
    # --------------------------------------------------------
    timestamp_monotonic_raw = msg["time"].is_monotonic_increasing
    duplicate_timestamps_raw = msg["time"].duplicated().sum()

    # Convert timestamps to New York time
    local_time = (
        pd.to_datetime(msg["time"], unit="ns", utc=True)
        .dt.tz_convert("America/New_York")
    )

    minutes = local_time.dt.hour * 60 + local_time.dt.minute

    regular_mask = (
        (minutes >= 9 * 60 + 30) &
        (minutes < 16 * 60)
    )

    # Apply exactly the same positional mask to synchronized files
    regular_idx = np.flatnonzero(regular_mask.to_numpy())

    msg_reg = msg.iloc[regular_idx]
    ob_reg = ob.iloc[regular_idx]

    # --------------------------------------------------------
    # 3. Missing values
    # --------------------------------------------------------
    message_missing = int(msg_reg.isna().sum().sum())
    orderbook_missing = int(ob_reg.isna().sum().sum())

    # --------------------------------------------------------
    # 4. Exact duplicate MESSAGE rows
    #
    # Important:
    # duplicate timestamps are allowed;
    # here we check whether the ENTIRE message row is duplicated.
    # We intentionally do not treat repeated LOB states as
    # duplicates because the same valid book state may persist
    # across multiple events.
    # --------------------------------------------------------
    exact_duplicate_message_rows = int(
        msg_reg.duplicated().sum()
    )

    # --------------------------------------------------------
    # 5. Message price / size diagnostics
    # --------------------------------------------------------
    zero_message_prices = int(
        (msg_reg["price"] == 0).sum()
    )

    extreme_message_prices = int(
        (msg_reg["price"] >= 90_000_000).sum()
    )

    negative_message_sizes = int(
        (msg_reg["size"] < 0).sum()
    )

    # --------------------------------------------------------
    # 6. Level-1 LOB validity
    # --------------------------------------------------------
    best_ask = ob_reg["sell1"]
    best_bid = ob_reg["buy1"]

    crossed_books = int(
        (best_bid > best_ask).sum()
    )

    locked_books = int(
        (best_bid == best_ask).sum()
    )

    nonpositive_best_ask = int(
        (best_ask <= 0).sum()
    )

    nonpositive_best_bid = int(
        (best_bid <= 0).sum()
    )

    # --------------------------------------------------------
    # 7. Negative displayed LOB sizes
    # Zero is not automatically treated as an error here.
    # --------------------------------------------------------
    volume_cols = (
        [f"vsell{i}" for i in range(1, 11)] +
        [f"vbuy{i}" for i in range(1, 11)]
    )

    negative_lob_sizes = int(
        (ob_reg[volume_cols] < 0).sum().sum()
    )

    # --------------------------------------------------------
    # 8. Ten-level price ordering
    #
    # Ask prices should increase away from the best ask:
    # sell1 <= sell2 <= ... <= sell10
    #
    # Bid prices should decrease away from the best bid:
    # buy1 >= buy2 >= ... >= buy10
    # --------------------------------------------------------
    ask_order_violation_rows = np.zeros(len(ob_reg), dtype=bool)
    bid_order_violation_rows = np.zeros(len(ob_reg), dtype=bool)

    for level in range(1, 10):
        ask_order_violation_rows |= (
            ob_reg[f"sell{level}"].to_numpy() >
            ob_reg[f"sell{level + 1}"].to_numpy()
        )

        bid_order_violation_rows |= (
            ob_reg[f"buy{level}"].to_numpy() <
            ob_reg[f"buy{level + 1}"].to_numpy()
        )

    ask_order_violations = int(
        ask_order_violation_rows.sum()
    )

    bid_order_violations = int(
        bid_order_violation_rows.sum()
    )

    result = {
        "asset": asset,
        "date": date,

        "raw_message_rows": message_rows_raw,
        "raw_orderbook_rows": orderbook_rows_raw,
        "row_count_match": row_count_match,

        "regular_session_rows": len(msg_reg),

        "timestamp_monotonic": timestamp_monotonic_raw,
        "duplicate_timestamps": int(duplicate_timestamps_raw),

        "exact_duplicate_message_rows":
            exact_duplicate_message_rows,

        "message_missing": message_missing,
        "orderbook_missing": orderbook_missing,

        "zero_message_prices": zero_message_prices,
        "extreme_message_prices": extreme_message_prices,
        "negative_message_sizes": negative_message_sizes,

        "crossed_books": crossed_books,
        "locked_books": locked_books,

        "nonpositive_best_ask": nonpositive_best_ask,
        "nonpositive_best_bid": nonpositive_best_bid,
        "negative_lob_sizes": negative_lob_sizes,

        "ask_order_violations": ask_order_violations,
        "bid_order_violations": bid_order_violations,
    }

    # Free memory before next multi-million-row asset-day
    del msg, ob, msg_reg, ob_reg, local_time
    gc.collect()

    return result


audit_results = []

for asset in assets:
    for date in dates:
        print(f"Auditing {asset} {date}...")
        audit_results.append(
            audit_training_day(asset, date)
        )

training_audit_df = pd.DataFrame(audit_results)

display(training_audit_df)

In [ ]:
# Inspect the exact duplicate message rows found on 2026-03-16
# without modifying the dataset.

for asset in ["TSLA", "AMZN"]:

    date = "2026-03-16"

    msg, ob = load_regular_session(asset, date)

    dup_mask = msg.duplicated(keep=False)
    dup = msg.loc[dup_mask].copy()

    print("\n" + "=" * 60)
    print(asset, date)
    print("=" * 60)

    print("Regular-session rows:", len(msg))
    print("Rows participating in duplicate groups:", dup_mask.sum())
    print("Duplicated rows beyond first occurrence:",
          msg.duplicated().sum())

    print("\nDuplicate rows by event type:")
    print(dup["event_type"].value_counts())

    print("\nNumber of distinct timestamps containing duplicates:")
    print(dup["time"].nunique())

    print("\nFirst 20 duplicate rows:")
    display(
        dup.sort_values("time").head(20)
    )

### Training-Set Audit Conclusion

The automated audit across all training asset-days found no systematic structural data-quality issues during the regular trading session. Message and order-book row counts were aligned, timestamps were monotonic, and no missing values, crossed/locked books, nonpositive best prices, negative LOB sizes, or price-level ordering violations were observed.

A small number of exact duplicate-looking message rows were found on 2026-03-16 for TSLA and AMZN. These were overwhelmingly `Trade` events with non-unique identifiers (`order_id = 0`), so they cannot be confidently classified as erroneous duplicates and were therefore retained. Their frequency was also below approximately 0.2% of regular-session observations.


In [ ]:
asset_summary = summary_df.groupby("asset").agg({
    "rows": "mean",
    "events_per_sec": "mean",
    "executions_per_sec": "mean",
    "spread_mean": "mean",
    "spread_median": "mean",
    "mid_std": "mean",
    "l1_imbalance_std": "mean",
    "avg_bid_size": "mean",
    "avg_ask_size": "mean",
})

display(asset_summary)

In [ ]:
def inspect_intraday_activity(asset, date):
    path = (
        DATA_DIR / asset / date /
        f"message_{date}.parquet"
    )

    msg = pd.read_parquet(
        path,
        columns=["time", "event_type"]
    )

    # Unix timestamp -> UTC -> New York local time
    local_time = (
        pd.to_datetime(msg["time"], unit="ns", utc=True)
        .dt.tz_convert("America/New_York")
    )

    temp = pd.DataFrame({
        "time": local_time,
        "event_type": msg["event_type"]
    })

    temp["hour"] = temp["time"].dt.hour

    hourly = temp.groupby("hour").size()

    print(f"\n{asset} {date}")
    print("First:", local_time.iloc[0])
    print("Last: ", local_time.iloc[-1])
    print("\nEvents by hour:")
    print(hourly)

    return temp

In [ ]:
tsla_316 = inspect_intraday_activity("TSLA", "2026-03-16")
amzn_316 = inspect_intraday_activity("AMZN", "2026-03-16")
brkb_316 = inspect_intraday_activity("BRK.B", "2026-03-16")

In [ ]:
def regular_session_counts(asset, date):
    path = (
        DATA_DIR / asset / date /
        f"message_{date}.parquet"
    )

    msg = pd.read_parquet(path, columns=["time"])

    local_time = (
        pd.to_datetime(msg["time"], unit="ns", utc=True)
        .dt.tz_convert("America/New_York")
    )

    minutes = (
        local_time.dt.hour * 60 +
        local_time.dt.minute
    )

    # 09:30 <= time < 16:00
    regular = (
        (minutes >= 9 * 60 + 30) &
        (minutes < 16 * 60)
    )

    print(asset, date)
    print("Total:", len(msg))
    print("Regular session:", regular.sum())
    print("Outside regular:", (~regular).sum())
    print("Outside %:", round((~regular).mean() * 100, 2))

In [ ]:
regular_session_counts("TSLA", "2026-03-16")
regular_session_counts("AMZN", "2026-03-16")
regular_session_counts("BRK.B", "2026-03-16")

In [ ]:
def load_regular_session(asset, date, lob_columns=None):
    day_dir = DATA_DIR / asset / date

    message_path = day_dir / f"message_{date}.parquet"
    orderbook_path = day_dir / f"orderbook_{date}.parquet"

    msg = pd.read_parquet(message_path)

    if lob_columns is None:
        ob = pd.read_parquet(orderbook_path)
    else:
        ob = pd.read_parquet(orderbook_path, columns=lob_columns)

    # Convert timestamps to New York time
    local_time = (
        pd.to_datetime(msg["time"], unit="ns", utc=True)
        .dt.tz_convert("America/New_York")
    )

    minutes = local_time.dt.hour * 60 + local_time.dt.minute

    # Regular trading session: 09:30 <= t < 16:00
    mask = (
        (minutes >= 9 * 60 + 30) &
        (minutes < 16 * 60)
    )

    # Same mask applied to synchronized message and LOB rows
    msg = msg.loc[mask].reset_index(drop=True)
    ob = ob.loc[mask].reset_index(drop=True)

    return msg, ob

In [ ]:
msg_test, ob_test = load_regular_session(
    "TSLA",
    "2026-03-16",
    lob_columns=["sell1", "vsell1", "buy1", "vbuy1"]
)

print("Message rows:", len(msg_test))
print("Orderbook rows:", len(ob_test))

print(
    pd.to_datetime(msg_test["time"].iloc[0], unit="ns", utc=True)
      .tz_convert("America/New_York")
)

print(
    pd.to_datetime(msg_test["time"].iloc[-1], unit="ns", utc=True)
      .tz_convert("America/New_York")
)

In [ ]:
def sampling_diagnostics(asset, date):
    message_path = (
        DATA_DIR / asset / date /
        f"message_{date}.parquet"
    )

    msg = pd.read_parquet(
        message_path,
        columns=["time"]
    )

    t = pd.to_datetime(msg["time"], unit="ns", utc=True)
    local = t.dt.tz_convert("America/New_York")

    minutes = local.dt.hour * 60 + local.dt.minute

    mask = (
        (minutes >= 9 * 60 + 30) &
        (minutes < 16 * 60)
    )

    local = local[mask]

    # Series with one count per event
    events = pd.Series(1, index=pd.DatetimeIndex(local))

    results = []

    for freq in ["100ms", "250ms", "500ms", "1s"]:
        counts = events.resample(freq).sum().fillna(0)

        results.append({
            "asset": asset,
            "frequency": freq,
            "bins": len(counts),
            "mean_events_per_bin": counts.mean(),
            "median_events_per_bin": counts.median(),
            "empty_bin_pct": (counts == 0).mean() * 100,
            "p95_events_per_bin": counts.quantile(0.95)
        })

    return pd.DataFrame(results)

In [ ]:
sampling_results = pd.concat([
    sampling_diagnostics("TSLA", "2026-03-18"),
    sampling_diagnostics("AMZN", "2026-03-18"),
    sampling_diagnostics("BRK.B", "2026-03-18")
], ignore_index=True)

display(sampling_results)

## 500 ms Calendar-Time Representation

The raw message and order-book data are event-driven, meaning observations arrive at
irregular times and the number of events occurring within a fixed amount of real time
varies substantially across assets and throughout the trading day. Since the forecasting
targets in this study are defined in terms of real elapsed time, the raw event stream is
converted to a regular calendar-time representation.

Candidate sampling frequencies of 100 ms, 250 ms, 500 ms, and 1 s were compared using
the number of observations per bin and, in particular, the proportion of intervals
containing no message events. The diagnostics showed that very short intervals are
relatively sparse even for the highly active assets and extremely sparse for BRK.B.
For example, at 100 ms approximately 13.0% of TSLA bins, 17.1% of AMZN bins, and
84.6% of BRK.B bins contained no events. At 500 ms these values decreased to
approximately 0.14%, 0.46%, and 53.0%, respectively.

A 500 ms grid was therefore selected as a compromise between temporal resolution,
data sparsity, and computational size. A finer 100 ms or 250 ms representation would
produce substantially more observations and many inactive intervals, particularly for
BRK.B, while a 1 s grid would reduce sparsity further but would also aggregate more of
the short-lived order-book dynamics that may be relevant for near-term forecasting.
The 500 ms representation produces 46,800 observations during a standard 6.5-hour
regular trading session.

The two data modalities are resampled according to their different interpretations.
For the limit order book, which represents the current market state, the last observed
book state within each 500 ms interval is retained. If no new book observation is
available in an interval, the most recently observed state is carried forward. Message
data instead represent discrete market events, so additions, deletions, executions, and
associated volumes are aggregated over all events occurring within each interval.

Using a common 500 ms base grid also gives all assets a consistent real-time
representation and allows forecasting horizons to be expressed directly in fixed numbers
of steps (e.g., 500 ms = 1 step, 2 s = 4 steps, and 10 s = 20 steps). Importantly, the
common sampling frequency does not imply that the same forecasting horizons must be used
for every asset. The substantially lower activity of BRK.B is considered separately in
the subsequent horizon-selection analysis.

In [ ]:
msg, ob = load_regular_session(
    "TSLA",
    "2026-03-18"
)

# Attach timestamp to the order book
time_index = pd.to_datetime(
    msg["time"],
    unit="ns",
    utc=True
).dt.tz_convert("America/New_York")

ob_temp = ob.copy()
ob_temp.index = pd.DatetimeIndex(time_index)

# Last observed book state in every 500 ms interval
lob_500ms = ob_temp.resample("500ms").last()

# If no book update occurs in a bin, carry forward
# the most recently known state
lob_500ms = lob_500ms.ffill()

print(lob_500ms.shape)
display(lob_500ms.head())

In [ ]:
price_cols = []

for level in range(1, 11):
    price_cols.extend([
        f"sell{level}",
        f"buy{level}"
    ])

lob_500ms[price_cols] = (
    lob_500ms[price_cols] / 1e4
)

In [ ]:
display(
    lob_500ms[
        ["sell1", "vsell1", "buy1", "vbuy1"]
    ].head()
)

## Exploratory Market-Microstructure Features

The 500 ms representation is used to construct interpretable predictors from
both the limit order book and message stream. LOB features summarize price,
spread, depth, imbalance, and microprice information, while message-based
features capture order additions, deletions, executions, and signed book flow.

These features provide the basis for the more systematic feature-engineering
pipeline implemented in the next notebook.

In [ ]:
lob_500ms["mid_price"] = (
    lob_500ms["sell1"] +
    lob_500ms["buy1"]
) / 2

lob_500ms["spread"] = (
    lob_500ms["sell1"] -
    lob_500ms["buy1"]
)

lob_500ms["imbalance_l1"] = (
    lob_500ms["vbuy1"] -
    lob_500ms["vsell1"]
) / (
    lob_500ms["vbuy1"] +
    lob_500ms["vsell1"]
)

In [ ]:
bid_depth_5 = sum(
    lob_500ms[f"vbuy{i}"]
    for i in range(1, 6)
)

ask_depth_5 = sum(
    lob_500ms[f"vsell{i}"]
    for i in range(1, 6)
)

lob_500ms["bid_depth_5"] = bid_depth_5
lob_500ms["ask_depth_5"] = ask_depth_5

lob_500ms["imbalance_l5"] = (
    bid_depth_5 - ask_depth_5
) / (
    bid_depth_5 + ask_depth_5
)

In [ ]:
lob_500ms["microprice"] = (
    lob_500ms["sell1"] * lob_500ms["vbuy1"]
    +
    lob_500ms["buy1"] * lob_500ms["vsell1"]
) / (
    lob_500ms["vbuy1"] +
    lob_500ms["vsell1"]
)

lob_500ms["microprice_deviation"] = (
    lob_500ms["microprice"] -
    lob_500ms["mid_price"]
)

In [ ]:
feature_cols = [
    "mid_price",
    "spread",
    "imbalance_l1",
    "bid_depth_5",
    "ask_depth_5",
    "imbalance_l5",
    "microprice",
    "microprice_deviation",
]

display(
    lob_500ms[feature_cols].describe().T
)

In [ ]:
msg_temp = msg.copy()

msg_temp.index = pd.DatetimeIndex(
    pd.to_datetime(
        msg_temp["time"],
        unit="ns",
        utc=True
    ).dt.tz_convert("America/New_York")
)

display(msg_temp.head())

In [ ]:
msg_temp["is_add"] = (
    msg_temp["event_type"] == "AddOrder"
).astype(int)

msg_temp["is_delete"] = (
    msg_temp["event_type"] == "OrderDelete"
).astype(int)

msg_temp["is_execution"] = (
    msg_temp["event_type"] == "OrderExecuted"
).astype(int)

msg_temp["is_buy_side"] = (
    msg_temp["direction"] == "Buy"
).astype(int)

msg_temp["is_sell_side"] = (
    msg_temp["direction"] == "Sell"
).astype(int)

In [ ]:
msg_temp["buy_add_volume"] = np.where(
    (msg_temp["event_type"] == "AddOrder") &
    (msg_temp["direction"] == "Buy"),
    msg_temp["size"],
    0
)

msg_temp["sell_add_volume"] = np.where(
    (msg_temp["event_type"] == "AddOrder") &
    (msg_temp["direction"] == "Sell"),
    msg_temp["size"],
    0
)

msg_temp["buy_delete_volume"] = np.where(
    (msg_temp["event_type"] == "OrderDelete") &
    (msg_temp["direction"] == "Buy"),
    msg_temp["size"],
    0
)

msg_temp["sell_delete_volume"] = np.where(
    (msg_temp["event_type"] == "OrderDelete") &
    (msg_temp["direction"] == "Sell"),
    msg_temp["size"],
    0
)

msg_temp["executed_buy_side_volume"] = np.where(
    (msg_temp["event_type"] == "OrderExecuted") &
    (msg_temp["direction"] == "Buy"),
    msg_temp["size"],
    0
)

msg_temp["executed_sell_side_volume"] = np.where(
    (msg_temp["event_type"] == "OrderExecuted") &
    (msg_temp["direction"] == "Sell"),
    msg_temp["size"],
    0
)

In [ ]:
flow_500ms = msg_temp.resample("500ms").agg(
    event_count=("event_type", "size"),
    add_count=("is_add", "sum"),
    delete_count=("is_delete", "sum"),
    execution_count=("is_execution", "sum"),

    buy_add_volume=("buy_add_volume", "sum"),
    sell_add_volume=("sell_add_volume", "sum"),

    buy_delete_volume=("buy_delete_volume", "sum"),
    sell_delete_volume=("sell_delete_volume", "sum"),

    executed_buy_side_volume=("executed_buy_side_volume", "sum"),
    executed_sell_side_volume=("executed_sell_side_volume", "sum"),
)

display(flow_500ms.head())
print(flow_500ms.shape)

In [ ]:
flow_500ms["net_book_flow"] = (
    flow_500ms["buy_add_volume"]
    - flow_500ms["buy_delete_volume"]
    - flow_500ms["sell_add_volume"]
    + flow_500ms["sell_delete_volume"]
)

In [ ]:
total_book_flow = (
    flow_500ms["buy_add_volume"]
    + flow_500ms["buy_delete_volume"]
    + flow_500ms["sell_add_volume"]
    + flow_500ms["sell_delete_volume"]
)

flow_500ms["normalized_book_flow"] = (
    flow_500ms["net_book_flow"] /
    total_book_flow.replace(0, np.nan)
).fillna(0)

In [ ]:
total_executed_volume = (
    flow_500ms["executed_buy_side_volume"]
    +
    flow_500ms["executed_sell_side_volume"]
)

flow_500ms["execution_imbalance"] = (
    flow_500ms["executed_sell_side_volume"]
    -
    flow_500ms["executed_buy_side_volume"]
) / total_executed_volume.replace(0, np.nan)

flow_500ms["execution_imbalance"] = (
    flow_500ms["execution_imbalance"].fillna(0)
)

In [ ]:
data_500ms = lob_500ms.join(
    flow_500ms,
    how="inner"
)

print("LOB:", lob_500ms.shape)
print("Flow:", flow_500ms.shape)
print("Combined:", data_500ms.shape)

In [ ]:
selected_cols = [
    "mid_price",
    "spread",
    "imbalance_l1",
    "imbalance_l5",
    "microprice_deviation",
    "event_count",
    "add_count",
    "delete_count",
    "execution_count",
    "normalized_book_flow",
    "execution_imbalance",
]

display(data_500ms[selected_cols].head(10))

In [ ]:
flow_features = [
    "event_count",
    "add_count",
    "delete_count",
    "execution_count",
    "normalized_book_flow",
    "execution_imbalance",
]

display(
    data_500ms[flow_features]
    .describe()
    .T
)

In [ ]:
print(
    "Bins without executions:",
    (data_500ms["execution_count"] == 0).mean() * 100,
    "%"
)

## Backward-Looking Temporal Features

Recent returns, volatility, and activity rates are added to capture short-term
temporal market dynamics. These predictors are constructed using only current
and past observations, so future information is not introduced into the
feature set.

In [ ]:
# Past returns: strictly backward-looking
data_500ms["return_500ms"] = np.log(
    data_500ms["mid_price"] /
    data_500ms["mid_price"].shift(1)
)

data_500ms["return_1s"] = np.log(
    data_500ms["mid_price"] /
    data_500ms["mid_price"].shift(2)
)

data_500ms["return_5s"] = np.log(
    data_500ms["mid_price"] /
    data_500ms["mid_price"].shift(10)
)

data_500ms["return_10s"] = np.log(
    data_500ms["mid_price"] /
    data_500ms["mid_price"].shift(20)
)

In [ ]:
data_500ms["volatility_5s"] = (
    data_500ms["return_500ms"]
    .rolling(10)
    .std()
)

data_500ms["volatility_30s"] = (
    data_500ms["return_500ms"]
    .rolling(60)
    .std()
)

In [ ]:
data_500ms["event_rate_5s"] = (
    data_500ms["event_count"]
    .rolling(10)
    .mean()
)

data_500ms["execution_rate_5s"] = (
    data_500ms["execution_count"]
    .rolling(10)
    .mean()
)

In [ ]:
data_500ms["avg_events_5s"] = (
    data_500ms["event_count"]
    .rolling(10)
    .mean()
)

data_500ms["avg_executions_5s"] = (
    data_500ms["execution_count"]
    .rolling(10)
    .mean()
)

## Forecast-Horizon Diagnostics

Candidate horizons from 500 ms to 30 s are compared using future-return
distributions and simple predictability diagnostics. The objective is to choose
meaningfully different real-time horizons while accounting for differences in
asset activity and short-horizon price behavior.

The resulting diagnostics motivate the asset-specific horizon sets used in the
downstream modeling experiments.

In [ ]:
candidate_horizons = {
    "500ms": 1,
    "1s": 2,
    "2s": 4,
    "5s": 10,
    "10s": 20,
    "30s": 60,
}

horizon_stats = []

for name, steps in candidate_horizons.items():

    future_return = np.log(
        data_500ms["mid_price"].shift(-steps) /
        data_500ms["mid_price"]
    )

    horizon_stats.append({
        "horizon": name,
        "mean": future_return.mean(),
        "std": future_return.std(),
        "mean_abs_return": future_return.abs().mean(),
        "median_abs_return": future_return.abs().median(),
        "zero_return_pct": (future_return == 0).mean() * 100,
        "p95_abs_return": future_return.abs().quantile(0.95),
    })

horizon_stats_df = pd.DataFrame(horizon_stats)

display(horizon_stats_df)

In [ ]:
predictability = []

for name, steps in candidate_horizons.items():

    future_return = np.log(
        data_500ms["mid_price"].shift(-steps) /
        data_500ms["mid_price"]
    )

    valid = (
        data_500ms["imbalance_l1"].notna() &
        future_return.notna()
    )

    corr_l1 = data_500ms.loc[
        valid, "imbalance_l1"
    ].corr(future_return[valid])

    corr_l5 = data_500ms.loc[
        valid, "imbalance_l5"
    ].corr(future_return[valid])

    corr_micro = data_500ms.loc[
        valid, "microprice_deviation"
    ].corr(future_return[valid])

    corr_flow = data_500ms.loc[
        valid, "normalized_book_flow"
    ].corr(future_return[valid])

    predictability.append({
        "horizon": name,
        "L1_imbalance": corr_l1,
        "L5_imbalance": corr_l5,
        "microprice_deviation": corr_micro,
        "book_flow": corr_flow,
    })

predictability_df = pd.DataFrame(predictability)

display(predictability_df)

In [ ]:
def build_500ms_data(asset, date):
    # -------------------------
    # 1. Load regular session
    # -------------------------
    msg, ob = load_regular_session(asset, date)

    time_index = pd.DatetimeIndex(
        pd.to_datetime(msg["time"], unit="ns", utc=True)
        .dt.tz_convert("America/New_York")
    )

    # -------------------------
    # 2. LOB state
    # -------------------------
    ob = ob.copy()
    ob.index = time_index

    lob = ob.resample("500ms").last().ffill()

    # Convert prices
    price_cols = []
    for level in range(1, 11):
        price_cols.extend([f"sell{level}", f"buy{level}"])

    lob[price_cols] = lob[price_cols] / 1e4

    # Basic LOB features
    lob["mid_price"] = (lob["sell1"] + lob["buy1"]) / 2
    lob["spread"] = lob["sell1"] - lob["buy1"]

    lob["imbalance_l1"] = (
        (lob["vbuy1"] - lob["vsell1"]) /
        (lob["vbuy1"] + lob["vsell1"])
    )

    bid_depth_5 = sum(lob[f"vbuy{i}"] for i in range(1, 6))
    ask_depth_5 = sum(lob[f"vsell{i}"] for i in range(1, 6))

    lob["bid_depth_5"] = bid_depth_5
    lob["ask_depth_5"] = ask_depth_5

    lob["imbalance_l5"] = (
        (bid_depth_5 - ask_depth_5) /
        (bid_depth_5 + ask_depth_5)
    )

    lob["microprice"] = (
        lob["sell1"] * lob["vbuy1"]
        + lob["buy1"] * lob["vsell1"]
    ) / (
        lob["vbuy1"] + lob["vsell1"]
    )

    lob["microprice_deviation"] = (
        lob["microprice"] - lob["mid_price"]
    )

    # -------------------------
    # 3. Message flow
    # -------------------------
    msg = msg.copy()
    msg.index = time_index

    msg["is_add"] = (msg["event_type"] == "AddOrder").astype(int)
    msg["is_delete"] = (msg["event_type"] == "OrderDelete").astype(int)
    msg["is_execution"] = (
        msg["event_type"] == "OrderExecuted"
    ).astype(int)

    definitions = {
        "buy_add_volume": ("AddOrder", "Buy"),
        "sell_add_volume": ("AddOrder", "Sell"),
        "buy_delete_volume": ("OrderDelete", "Buy"),
        "sell_delete_volume": ("OrderDelete", "Sell"),
        "executed_buy_side_volume": ("OrderExecuted", "Buy"),
        "executed_sell_side_volume": ("OrderExecuted", "Sell"),
    }

    for col, (event, side) in definitions.items():
        msg[col] = np.where(
            (msg["event_type"] == event) &
            (msg["direction"] == side),
            msg["size"],
            0
        )

    flow = msg.resample("500ms").agg(
        event_count=("event_type", "size"),
        add_count=("is_add", "sum"),
        delete_count=("is_delete", "sum"),
        execution_count=("is_execution", "sum"),

        buy_add_volume=("buy_add_volume", "sum"),
        sell_add_volume=("sell_add_volume", "sum"),
        buy_delete_volume=("buy_delete_volume", "sum"),
        sell_delete_volume=("sell_delete_volume", "sum"),

        executed_buy_side_volume=(
            "executed_buy_side_volume", "sum"
        ),
        executed_sell_side_volume=(
            "executed_sell_side_volume", "sum"
        ),
    )

    total_flow = (
        flow["buy_add_volume"]
        + flow["sell_add_volume"]
        + flow["buy_delete_volume"]
        + flow["sell_delete_volume"]
    )

    flow["net_book_flow"] = (
        flow["buy_add_volume"]
        - flow["buy_delete_volume"]
        - flow["sell_add_volume"]
        + flow["sell_delete_volume"]
    )

    flow["normalized_book_flow"] = (
        flow["net_book_flow"] /
        total_flow.replace(0, np.nan)
    ).fillna(0)

    # -------------------------
    # 4. Combine
    # -------------------------
    data = lob.join(flow, how="inner")

    return data
    

## Cross-Asset Horizon Comparison

The candidate-horizon diagnostics are evaluated across all three training days
(March 16--18) for each asset. Future returns are constructed separately within
each trading day before observations are pooled, preventing targets from crossing
overnight boundaries.

Using all available training days provides a more robust basis for horizon
selection than relying on a single representative day, while still avoiding any
use of validation or test information. The diagnostics compare return variability,
the frequency of zero future returns, and simple associations between future
returns and market-microstructure predictors across candidate horizons from
500 ms to 30 s.

Because the assets differ substantially in trading activity and short-horizon
price behavior, a single horizon set is not imposed on every asset. The selected
horizons are determined from these training-period diagnostics and fixed before
the downstream validation and test comparisons.

In [ ]:
# ============================================================
# Cross-asset horizon diagnostics using all training days
# March 16--18
#
# Future returns are computed independently within each day
# so that targets never cross overnight/day boundaries.
# ============================================================

training_dates = [
    "2026-03-16",
    "2026-03-17",
    "2026-03-18",
]

candidate_horizons = {
    "500ms": 1,
    "1s": 2,
    "2s": 4,
    "5s": 10,
    "10s": 20,
    "30s": 60,
}

horizon_assets = ["TSLA", "AMZN", "BRK.B"]

pooled = {}
daily_results = []

for asset in horizon_assets:

    for date in training_dates:

        print(f"Processing horizon diagnostics: {asset} {date}")

        data = build_500ms_data(asset, date)

        for horizon, steps in candidate_horizons.items():

            # Future return is computed within this asset-day only
            future_return = np.log(
                data["mid_price"].shift(-steps) /
                data["mid_price"]
            )

            valid = (
                future_return.notna()
                & data["imbalance_l1"].notna()
                & data["imbalance_l5"].notna()
                & data["microprice_deviation"].notna()
                & data["normalized_book_flow"].notna()
            )

            temp = pd.DataFrame({
                "future_return":
                    future_return.loc[valid].to_numpy(),

                "imbalance_l1":
                    data.loc[valid, "imbalance_l1"].to_numpy(),

                "imbalance_l5":
                    data.loc[valid, "imbalance_l5"].to_numpy(),

                "microprice_deviation":
                    data.loc[valid, "microprice_deviation"].to_numpy(),

                "normalized_book_flow":
                    data.loc[valid, "normalized_book_flow"].to_numpy(),
            })

            pooled.setdefault(
                (asset, horizon), []
            ).append(temp)

            daily_results.append({
                "asset": asset,
                "date": date,
                "horizon": horizon,

                "std_return":
                    temp["future_return"].std(),

                "mean_abs_return":
                    temp["future_return"].abs().mean(),

                "zero_return_pct":
                    (temp["future_return"] == 0).mean() * 100,

                "corr_L1":
                    temp["imbalance_l1"].corr(
                        temp["future_return"]
                    ),

                "corr_L5":
                    temp["imbalance_l5"].corr(
                        temp["future_return"]
                    ),

                "corr_micro":
                    temp["microprice_deviation"].corr(
                        temp["future_return"]
                    ),

                "corr_flow":
                    temp["normalized_book_flow"].corr(
                        temp["future_return"]
                    ),
            })

daily_horizon_diagnostics = pd.DataFrame(daily_results)

In [ ]:
# Pool observations across the three training days
# after constructing future returns independently within each day.

pooled_results = []

for (asset, horizon), frames in pooled.items():

    temp = pd.concat(
        frames,
        ignore_index=True
    )

    pooled_results.append({
        "asset": asset,
        "horizon": horizon,

        "std_return":
            temp["future_return"].std(),

        "mean_abs_return":
            temp["future_return"].abs().mean(),

        "zero_return_pct":
            (temp["future_return"] == 0).mean() * 100,

        "corr_L1":
            temp["imbalance_l1"].corr(
                temp["future_return"]
            ),

        "corr_L5":
            temp["imbalance_l5"].corr(
                temp["future_return"]
            ),

        "corr_micro":
            temp["microprice_deviation"].corr(
                temp["future_return"]
            ),

        "corr_flow":
            temp["normalized_book_flow"].corr(
                temp["future_return"]
            ),
    })

cross_asset_horizons = pd.DataFrame(pooled_results)

display(cross_asset_horizons)

In [ ]:
display(
    daily_horizon_diagnostics[
        [
            "asset",
            "date",
            "horizon",
            "zero_return_pct",
            "corr_L1",
            "corr_micro",
        ]
    ]
)

### Selected Forecast Horizons

The pooled March 16--18 training diagnostics confirm substantial differences in
short-horizon price behavior across assets.

For TSLA, the 500 ms future return is zero in approximately 23.5% of observations,
and the strongest imbalance and microprice associations occur at short horizons and
decay as the forecast horizon increases. AMZN exhibits a similar pattern: approximately
38.4% of 500 ms returns are zero, while level-1 imbalance and microprice deviation show
their strongest associations at sub-second and short-second horizons.

BRK.B is substantially less active. Approximately 88.1% of its 500 ms future returns
are zero, compared with 66.1% at 2 s, 41.8% at 5 s, 22.3% at 10 s, and 4.3% at 30 s.
This behavior is consistent across all three training days rather than being driven by
a single session. Longer horizons therefore provide a more informative target for BRK.B.

Based on these training-period diagnostics, the forecasting horizons are fixed as:

- **TSLA:** 500 ms, 2 s, and 10 s
- **AMZN:** 500 ms, 2 s, and 10 s
- **BRK.B:** 5 s, 10 s, and 30 s

The three horizons are intended to probe different short-term market-response
scales rather than three arbitrary prediction intervals. For TSLA and AMZN,
500 ms captures the immediate response of the highly active order book, 2 s
provides an intermediate horizon after some of the sub-second microstructure
noise has dissipated, and 10 s tests whether the short-horizon imbalance and
microprice signals persist or decay over a longer interval.

BRK.B is substantially less active, so a 500 ms or 2 s target is dominated by
unchanged mid-prices. Its 5 s horizon serves as the shortest reasonably
informative target, 10 s provides an intermediate scale, and 30 s provides a
longer short-term horizon where non-zero price movements are sufficiently common.
Thus the horizon sets reflect both the empirical diagnostics and the different
market activity of the assets.

The same horizon choices are then used throughout downstream validation and held-out
test comparisons. No validation or test data are used to select these horizons.

In [ ]:
def add_temporal_features(data):
    data = data.copy()

    # -------------------------
    # Past returns
    # -------------------------
    data["return_500ms"] = np.log(
        data["mid_price"] /
        data["mid_price"].shift(1)
    )

    data["return_1s"] = np.log(
        data["mid_price"] /
        data["mid_price"].shift(2)
    )

    data["return_2s"] = np.log(
        data["mid_price"] /
        data["mid_price"].shift(4)
    )

    data["return_5s"] = np.log(
        data["mid_price"] /
        data["mid_price"].shift(10)
    )

    data["return_10s"] = np.log(
        data["mid_price"] /
        data["mid_price"].shift(20)
    )

    # -------------------------
    # Rolling volatility
    # -------------------------
    r = data["return_500ms"]

    data["volatility_5s"] = r.rolling(10).std()
    data["volatility_30s"] = r.rolling(60).std()

    # -------------------------
    # Recent market activity
    # -------------------------
    data["avg_events_5s"] = (
        data["event_count"]
        .rolling(10)
        .mean()
    )

    data["avg_executions_5s"] = (
        data["execution_count"]
        .rolling(10)
        .mean()
    )

    return data

In [ ]:
def add_targets(data, asset):
    data = data.copy()

    if asset in ["TSLA", "AMZN"]:
        horizons = {
            "500ms": 1,
            "2s": 4,
            "10s": 20,
        }
    else:  # BRK.B
        horizons = {
            "5s": 10,
            "10s": 20,
            "30s": 60,
        }

    for name, steps in horizons.items():
        data[f"target_{name}"] = np.log(
            data["mid_price"].shift(-steps) /
            data["mid_price"]
        )

    return data

In [ ]:
data = build_500ms_data(asset, date)
data = add_temporal_features(data)
data = add_targets(data, asset)